# 🌿 GPU-Accelerated Vegan Recipe Batch Generator
Run this on Google Colab with a T4 GPU. This will bypass your Hugging Face API entirely and run the model directly on the Colab GPU for blazing fast speeds.

In [ ]:
# 1. Install Dependencies
!pip install unsloth -q
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install "trl==0.24.0" peft accelerate bitsandbytes sentence-transformers scipy -q


In [ ]:
# 2. Clone repository to get datasets and api logic
!git clone https://github.com/dn74iiit/ratatouille-app.git
%cd ratatouille-app


In [ ]:
# 3. Load the Model directly onto GPU
import torch
from unsloth import FastLanguageModel

# I set this to your V8 model since it's confirmed public, but you can change it to your V10 model!
MODEL_ID = "nd1490/ratatouille-llama3-3b-v8-50k" 

print(f"Loading {MODEL_ID} on GPU...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=512,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("Model ready!")


In [ ]:
# 4. Define the 25 test cases and Generation Loop
import csv
import time

# We import your substitution engine logic directly from your API file
# (We bypass the FastAPI endpoints and just use the python functions!)
from api import get_vegan_blueprint

test_cases = [
    {"ingredients": ["chicken", "rice", "onion", "tomato"], "budget": 200, "name": "Chicken Biryani"},
    {"ingredients": ["mutton", "yogurt", "garlic", "ginger"], "budget": 250, "name": "Mutton Curry"},
    {"ingredients": ["egg", "onion", "green chili", "tomato"], "budget": 50, "name": "Egg Bhurji"},
    {"ingredients": ["paneer", "spinach", "cream", "garlic"], "budget": 150, "name": "Palak Paneer"},
    {"ingredients": ["chicken", "butter", "cream", "tomato puree"], "budget": 300, "name": "Butter Chicken"},
    {"ingredients": ["fish", "mustard oil", "mustard seeds", "turmeric"], "budget": 180, "name": "Fish Curry"},
    {"ingredients": ["pork", "vinegar", "garlic", "chili"], "budget": 200, "name": "Pork Vindaloo"},
    {"ingredients": ["egg", "rice", "soy sauce", "carrot"], "budget": 100, "name": "Egg Fried Rice"},
    {"ingredients": ["beef", "potato", "onion", "carrot"], "budget": 220, "name": "Beef Stew"},
    {"ingredients": ["chicken", "coconut milk", "lemongrass", "galangal"], "budget": 250, "name": "Chicken Thai Curry"},
    {"ingredients": ["shrimp", "garlic", "butter", "lemon"], "budget": 300, "name": "Garlic Butter Shrimp"},
    {"ingredients": ["paneer", "capsicum", "onion", "tomato"], "budget": 150, "name": "Kadai Paneer"},
    {"ingredients": ["minced meat", "pea", "onion", "tomato"], "budget": 200, "name": "Keema Matar"},
    {"ingredients": ["chicken", "cashew", "cream", "onion"], "budget": 280, "name": "Chicken Korma"},
    {"ingredients": ["egg", "potato", "onion", "tomato"], "budget": 80, "name": "Egg Potato Curry"},
    {"ingredients": ["mutton", "rice", "mint", "yogurt"], "budget": 350, "name": "Mutton Biryani"},
    {"ingredients": ["crab", "coconut", "chili", "tamarind"], "budget": 400, "name": "Crab Curry"},
    {"ingredients": ["chicken", "noodle", "soy sauce", "cabbage"], "budget": 120, "name": "Chicken Hakka Noodles"},
    {"ingredients": ["paneer", "pea", "tomato", "onion"], "budget": 140, "name": "Matar Paneer"},
    {"ingredients": ["beef", "bun", "cheese", "lettuce"], "budget": 150, "name": "Beef Burger"},
    {"ingredients": ["lamb", "rosemary", "potato", "garlic"], "budget": 300, "name": "Roast Lamb"},
    {"ingredients": ["squid", "flour", "oil", "lemon"], "budget": 250, "name": "Fried Calamari"},
    {"ingredients": ["chicken", "mushroom", "cream", "pasta"], "budget": 200, "name": "Chicken Mushroom Pasta"},
    {"ingredients": ["egg", "flour", "sugar", "butter"], "budget": 100, "name": "Egg Cake"},
    {"ingredients": ["salmon", "asparagus", "lemon", "butter"], "budget": 350, "name": "Grilled Salmon"}
]

csv_file = "GPU_vegan_recipe_results.csv"

# Function to format the prompt exactly how your model was trained
def build_vegan_prompt(ingredients_list):
    ingr_text = "\n".join([f"- {i}" for i in ingredients_list])
    return (
        f"### INGREDIENTS:\n{ingr_text}\n"
        f"### TITLE:\n"
        f"### DIRECTIONS:\n"
    )

with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(["Original Dish", "Original Ingredients", "Veganized Ingredients", "Generated Recipe"])

    for i, test in enumerate(test_cases):
        print(f"[{i+1}/25] Generating {test['name']}...")
        
        # 1. Run local vegan substitution logic
        try:
            blueprint = get_vegan_blueprint(test['ingredients'], archetype="Curry")
            veganized_ingredients = [res['best_vegan_substitute'] for res in blueprint if res['status'] == 'substituted']
            veganized_ingredients += [res['original_ingredient'] for res in blueprint if res['status'] == 'already_vegan']
            
            # Extract spice bridges to compensate for missing flavor profiles
            for res in blueprint:
                if res['status'] == 'substituted':
                    for bridge in res['compensation_blueprint']['spice_bridge']:
                        veganized_ingredients.append(bridge['spice'])
        except Exception as e:
            print(f"Error in substitution logic: {e}")
            continue
            
        vegan_ing_str = ", ".join(veganized_ingredients)
        
        # 2. Format prompt
        prompt = build_vegan_prompt(veganized_ingredients)
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        
        # 3. Generate recipe locally on GPU
        start_time = time.time()
        out = model.generate(
            **inputs, 
            max_new_tokens=400, 
            temperature=0.7,
            do_sample=True, 
            eos_token_id=tokenizer.eos_token_id
        )
        recipe_text = tokenizer.decode(out[0], skip_special_tokens=True)
        print(f"  -> Done in {time.time() - start_time:.1f}s")
        
        writer.writerow([test['name'], ", ".join(test['ingredients']), vegan_ing_str, recipe_text])
        file.flush()

print(f"\nAll recipes processed and saved to {csv_file}!")
